# 实验七 · 向量归一化与屏障

**所属**：《并行计算》第四章 · Pthread 多线程编程　|　**难度**：⭐⭐⭐⭐ 综合　|　**预计时长**：20–30 分钟

> **实验说明**
> 1. 前面几个实验中，线程之间的等待都是「一个等一个」。本实验面对一类新的同步需求：**全体线程互相等待**——所有线程都到达某个点之后，任何一个才能继续。这就是**屏障**（barrier）。
> 2. 本实验的核心不是归一化算法本身，而是**如何实现屏障**。程序把同一个屏障用四种方式实现，从错误版本出发，逐级递进到 POSIX 标准 API，最终目的是理解 `pthread_barrier_wait` 内部到底做了什么。
> 3. 四个版本共用同一段计算代码，仅在阶段之间调用的屏障不同，因此性能差异只可能来自屏障实现本身。
> 4. 请自上而下依次执行各单元格（Shift+Enter）。
> 5. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。

## 🎯 学习目标

完成本实验后，学生应能够：

- 说明**屏障**的语义，并区分它与互斥、与「一个线程等待另一个线程」的不同
- 解释为何「互斥」不能替代「同步」，即保护了共享变量仍不足以保证阶段间的正确顺序
- 用原子操作实现自旋屏障，并说明其中 **generation 计数器**为何不可省略
- 用互斥量与条件变量实现屏障，说明放行时为何必须用 `broadcast` 而非 `signal`
- 使用 POSIX 标准的 `pthread_barrier_t`，并说明它相比手写实现的取舍
- 说明屏障处的等待时间由最慢的线程决定，从而理解负载均衡对屏障密集型算法的意义

## 🗺️ 学习路径

1. **准备阶段**：理解多阶段计算模型，明确「阶段间依赖」这一新的同步需求
2. **概念辨析**：区分互斥与同步——互斥回答「不要同时动」，同步回答「等大家都做完」
3. **版本一（无屏障）**：阶段之间不同步，读到尚未完成计算的中间结果
4. **版本二（原子自旋屏障）**：手写屏障，理解 generation 计数器的作用
   → 优点：延迟低；缺点：等待期间满载占用 CPU
5. **版本三（条件变量屏障）**：等待线程真正睡眠，理解 `broadcast` 的必要性
6. **版本四（POSIX 标准屏障）**：一行 `pthread_barrier_wait` 取代二十行手写代码
7. **性能对比**：四种实现的取舍，以及屏障与负载均衡的关系

## 1. 背景与动机

许多算法天然分为**多个阶段**，后一阶段依赖前一阶段的**全部**结果。这类算法无法简单地按数据分块并行——因为在进入下一阶段之前，必须确认所有线程都已完成当前阶段。

以**向量归一化**为例。给定向量 $a$，要计算归一化结果 $b$，使得 $b$ 的各分量之和为 1：

$$b_i = \frac{a_i}{\sum_{j} a_j}$$

并行计算它，自然地分为三个阶段：

```
   阶段 1          屏障        阶段 2          屏障        阶段 3
  各线程计算    ──────────►  0 号线程汇总   ──────────►  各线程用全局和
  自己分块的                  各分块的局部和              归一化自己的分块
  局部和 partial              得到 global_sum             b[i] = a[i]/global_sum
```

**关键在于阶段之间的依赖**：

- 阶段 2 要汇总所有 `partial_sum[t]`，必须等**所有**线程的阶段 1 都完成；
- 阶段 3 要用 `global_sum` 做除法，必须等阶段 2 完成。

若某个线程跑得快，在别的线程还没算完阶段 1 时就冲进了阶段 2，它读到的就是一份**不完整**的数据。这正是屏障要解决的问题。

## 2. 屏障的语义

**屏障**（barrier）是一个同步点，其规则是：

> **所有参与线程必须全部到达屏障点，任何一个线程才能继续向下执行。**
> 先到达的线程阻塞等待，最后一个到达的线程负责放行全体。

```
   线程0  ──██──────────────►│ 继续
   线程1  ──████████─────────►│ 继续       ← 屏障：全部到齐才放行
   线程2  ──███────────────── │ 继续
   线程3  ──██████████████───►│ 继续
                              ↑
                         最后一个到达者放行全体
```

这与之前实验中的等待都不同：

<!--
| 同步形式 | 等待关系 | 引入实验 |
|---|---|---|
| 互斥量 | 一个进，其余等 | 实验三 |
| 信号量 / 条件变量 | 一个线程等另一个线程 | 实验五、实验六 |
| **屏障** | **全体互相等待** | **本实验** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">同步形式</th>
      <th style="text-align: left;">等待关系</th>
      <th style="text-align: left;">引入实验</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">互斥量</td>
      <td style="text-align: left;">一个进，其余等</td>
      <td style="text-align: left;">实验三</td>
    </tr>
    <tr>
      <td style="text-align: left;">信号量 / 条件变量</td>
      <td style="text-align: left;">一个线程等另一个线程</td>
      <td style="text-align: left;">实验五、实验六</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>屏障</strong></td>
      <td style="text-align: left;"><strong>全体互相等待</strong></td>
      <td style="text-align: left;"><strong>本实验</strong></td>
    </tr>
  </tbody>
</table>

### 💡 与 Amdahl 定律的联系

屏障处的等待时间，**由最慢的那个线程决定**。若各线程负载不均，快的线程只能空等——这段等待时间等效于**串行开销**，会直接压低加速比的上限。

因此，屏障密集的算法必须格外重视**负载均衡**：让每个线程分到的工作量尽可能相等，才能减少屏障处的空等。这一点在第 9 节会通过实测体现。

## 3. 互斥 ≠ 同步

这是本实验最重要的一个概念区分。

考虑阶段 1 到阶段 2 的衔接。一种常见的错误认识是：「只要给 `global_sum` 的累加加上互斥量，不就正确了吗？」

```c
// 阶段 1：各线程累加自己的局部和
pthread_mutex_lock(&mutex);
global_sum += local_sum;        // 互斥量保证这里不丢失更新
pthread_mutex_unlock(&mutex);

// 没有屏障！

// 阶段 2：直接使用 global_sum —— 但它可能还不完整
b[i] = a[i] / global_sum;
```

互斥量确实保证了 `global_sum` 的累加**不会丢失更新**——这一点是对的。但它**完全没有保证累加已经全部完成**。跑得快的线程可能在其他线程尚未贡献自己那份局部和时，就读走了一个未计算完的 `global_sum`，用错误的和去做归一化。

<!--
| | 互斥（Mutual Exclusion） | 同步（Synchronization） |
|---|---|---|
| 回答的问题 | 「不要**同时**动」 | 「等**大家都做完**」 |
| 典型工具 | 互斥量 | 屏障、条件变量 |
| 本实验中的作用 | 保证累加不丢失 | 保证阶段顺序正确 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">互斥（Mutual Exclusion）</th>
      <th style="text-align: left;">同步（Synchronization）</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">回答的问题</td>
      <td style="text-align: left;">「不要<strong>同时</strong>动」</td>
      <td style="text-align: left;">「等<strong>大家都做完</strong>」</td>
    </tr>
    <tr>
      <td style="text-align: left;">典型工具</td>
      <td style="text-align: left;">互斥量</td>
      <td style="text-align: left;">屏障、条件变量</td>
    </tr>
    <tr>
      <td style="text-align: left;">本实验中的作用</td>
      <td style="text-align: left;">保证累加不丢失</td>
      <td style="text-align: left;">保证阶段顺序正确</td>
    </tr>
  </tbody>
</table>

> **互斥回答「不要同时动」，同步回答「等大家都做完」。**
> 二者解决的是完全不同的问题，不能互相替代。

本实验的错误版本（版本一）恰恰只有「按阶段划分」而没有屏障，下面将看到它如何出错。

## 4. 环境准备

In [ ]:
import platform, subprocess, shutil, sys, os, re

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
NCPU = os.cpu_count()
print("CPU 核心:", NCPU)

if CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
elif NCPU == 1:
    print("\n⚠️  当前仅 1 个核心：线程只能分时轮转。")
    print(
        "    无屏障版本仍可能出错；自旋屏障会因线程数超过核心数而严重退化（见第 8 节）。"
    )
else:
    print(f"\n✅ 环境就绪：编译器可用，{NCPU} 核可用，可以开始实验！")


### 编译与运行工具函数

本实验的四个版本合并在一个程序中，一次运行即可全部对比。C11 原子操作、条件变量与 `pthread_barrier_t` 均由 `-lpthread` 链接。

In [2]:
SRC_DIR = "src_normalize"
os.makedirs(SRC_DIR, exist_ok=True)


def compile_c(src, out):
    """用全章统一选项编译一个源文件，成功返回可执行文件名，失败返回 None。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    cmd = f"{base} -O3 -fPIC -pthread -Wall -Wextra {src} -o {out} -lpthread -lm"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print("✅ 编译成功：", cmd)
        if r.stderr.strip():
            print(r.stderr.strip())
        return out
    print("❌ 编译失败：\n", r.stderr)
    return None


def run_bin(out, *args, echo=True, timeout=120):
    """运行可执行文件并返回其标准输出；echo=True 时同时打印。"""
    r = subprocess.run(
        ["./" + out] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if echo:
        print(r.stdout, end="")
        if r.returncode != 0 and r.stderr:
            print("STDERR:", r.stderr)
    return r.stdout


def parse_versions(text):
    """解析输出表，返回 [(版本名, 耗时ms, sum_b, 校验)]。"""
    rows = []
    for line in text.splitlines():
        m = re.match(
            r"^(\d\.\s+.+?)\s+([\d.]+)\s+(inf|nan|[\d.]+)\s+(PASS|FAIL)\s*$", line
        )
        if m:
            v = float("inf") if m.group(3) in ("inf", "nan") else float(m.group(3))
            rows.append((m.group(1).strip(), float(m.group(2)), v, m.group(4)))
    return rows


## 5. 算法结构：共用的三阶段计算

四个版本的计算部分**完全相同**，都调用同一个函数 `normalize_phases`，仅通过一个**函数指针**参数决定在阶段之间调用哪种屏障：

```c
static void normalize_phases(long my_rank, void (*barrier_fn)(void)) {
  // 阶段 1：初始化自己分块的数据
  for (long i = my_first_i; i < my_last_i; ++i) a[i] = (double)(i + 1);
  if (barrier_fn != NULL) barrier_fn();          // ← 屏障 1

  // 阶段 2：计算自己分块的局部和
  double local_sum = 0.0;
  for (long i = my_first_i; i < my_last_i; ++i) local_sum += a[i];
  partial_sum[my_rank] = local_sum;
  if (barrier_fn != NULL) barrier_fn();          // ← 屏障 2

  // 阶段 3：0 号线程汇总全局和
  if (my_rank == 0) {
    global_sum = 0.0;
    for (long t = 0; t < thread_count; ++t) global_sum += partial_sum[t];
  }
  if (barrier_fn != NULL) barrier_fn();          // ← 屏障 3

  // 阶段 4：各线程用全局和归一化自己的分块
  for (long i = my_first_i; i < my_last_i; ++i) b[i] = a[i] / global_sum;
}
```

这个设计有两个好处：

1. **公平性**：四个版本跑的是同一段计算代码，性能差异只来自屏障实现；
2. **清晰性**：传入 `NULL` 就得到无屏障的错误版本，屏障的作用一目了然。

程序里一共有**三个屏障点**，分别隔开四个阶段。其中屏障 2 是最关键的：阶段 3 汇总 `partial_sum` 之前，必须确保所有线程的阶段 2 都已写入。

> **正确性判据**：由于 $b_i = a_i / \sum a_j$，因此 $\sum b_i$ 必然**精确等于 1**。程序对结果求和，若与 1 的偏差超过 $10^{-5}$，或结果为 `inf`／`nan`，即判定为 `FAIL`。

In [ ]:
%%writefile {SRC_DIR}/pthread_normalization.c
#include <math.h>
#include <pthread.h>
#include <stdatomic.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define MAX_THREADS 64

long thread_count = 0;
long n = 0;

double* a = NULL;
double* b = NULL;
double* partial_sum = NULL;
double global_sum = 0.0;

// Version 2: hand-built atomic spin barrier.
atomic_long atomic_arrived = 0;
atomic_long atomic_generation = 0;

// Version 3: mutex + condition variable barrier.
long cond_counter = 0;
long cond_generation = 0;
pthread_mutex_t barrier_mutex;
pthread_cond_t barrier_cond;

// Version 4: POSIX standard barrier.
pthread_barrier_t barrier;

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// ---------------------------------------------------------------------------
// Barrier implementations
// ---------------------------------------------------------------------------

static void busy_barrier_wait(void) {
  long my_gen = atomic_load(&atomic_generation);
  long old_count = atomic_fetch_add(&atomic_arrived, 1);

  if (old_count + 1 == thread_count) {
    atomic_store(&atomic_arrived, 0);
    atomic_fetch_add(&atomic_generation, 1);
  } else {
    while (atomic_load(&atomic_generation) == my_gen) {
      // spin: costs a full core per waiting thread
    }
  }
}

static void cond_barrier_wait(void) {
  pthread_mutex_lock(&barrier_mutex);
  long my_gen = cond_generation;
  ++cond_counter;

  if (cond_counter == thread_count) {
    cond_counter = 0;
    ++cond_generation;
    pthread_cond_broadcast(&barrier_cond);
  } else {
    while (cond_generation == my_gen) {
      pthread_cond_wait(&barrier_cond, &barrier_mutex);
    }
  }
  pthread_mutex_unlock(&barrier_mutex);
}

// ---------------------------------------------------------------------------
// The four thread functions. They differ only in which barrier they call.
// ---------------------------------------------------------------------------

// Computes the four phases, calling barrier_fn between them. Passing NULL
// omits the synchronization entirely, which reproduces the broken version.
static void normalize_phases(long my_rank, void (*barrier_fn)(void)) {
  long local_n = n / thread_count;
  long my_first_i = my_rank * local_n;
  long my_last_i = my_first_i + local_n;

  for (long i = my_first_i; i < my_last_i; ++i) {
    a[i] = (double)(i + 1);
  }
  if (barrier_fn != NULL) barrier_fn();

  double local_sum = 0.0;
  for (long i = my_first_i; i < my_last_i; ++i) {
    local_sum += a[i];
  }
  partial_sum[my_rank] = local_sum;
  if (barrier_fn != NULL) barrier_fn();

  if (my_rank == 0) {
    global_sum = 0.0;
    for (long t = 0; t < thread_count; ++t) {
      global_sum += partial_sum[t];
    }
  }
  if (barrier_fn != NULL) barrier_fn();

  for (long i = my_first_i; i < my_last_i; ++i) {
    b[i] = a[i] / global_sum;
  }
}

static void std_barrier_wait(void) { pthread_barrier_wait(&barrier); }

void* Thread_no_barrier(void* rank) {
  normalize_phases((long)rank, NULL);
  return NULL;
}

void* Thread_busy_barrier(void* rank) {
  normalize_phases((long)rank, busy_barrier_wait);
  return NULL;
}

void* Thread_cond_barrier(void* rank) {
  normalize_phases((long)rank, cond_barrier_wait);
  return NULL;
}

void* Thread_std_barrier(void* rank) {
  normalize_phases((long)rank, std_barrier_wait);
  return NULL;
}

// ---------------------------------------------------------------------------
// Benchmark driver
// ---------------------------------------------------------------------------

// Runs one version and reports its time and whether sum(b) == 1.
static void run_version(const char* name, void* (*worker)(void*),
                        pthread_t* handles) {
  global_sum = 0.0;
  for (long i = 0; i < n; ++i) b[i] = 0.0;
  atomic_store(&atomic_arrived, 0);
  atomic_store(&atomic_generation, 0);
  cond_counter = 0;
  cond_generation = 0;

  double start = get_time_ms();
  for (long t = 0; t < thread_count; ++t) {
    if (pthread_create(&handles[t], NULL, worker, (void*)t) != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      exit(1);
    }
  }
  for (long t = 0; t < thread_count; ++t) pthread_join(handles[t], NULL);
  double elapsed = get_time_ms() - start;

  double check_sum = 0.0;
  for (long i = 0; i < n; ++i) check_sum += b[i];

  const char* status =
      isfinite(check_sum) && fabs(check_sum - 1.0) < 1e-5 ? "PASS" : "FAIL";
  printf("%-28s %12.3f %14.6f %8s\n", name, elapsed, check_sum, status);
}

int main(int argc, char* argv[]) {
  if (argc != 3) {
    fprintf(stderr, "Usage: %s <thread_count> <n>\n", argv[0]);
    return 1;
  }

  thread_count = strtol(argv[1], NULL, 10);
  n = strtol(argv[2], NULL, 10);

  if (thread_count <= 0 || thread_count > MAX_THREADS) {
    fprintf(stderr, "Error: thread_count must be between 1 and %d\n",
            MAX_THREADS);
    return 1;
  }
  if (n <= 0 || n % thread_count != 0) {
    fprintf(stderr,
            "Error: n must be positive and divisible by thread_count\n");
    return 1;
  }

  a = malloc(n * sizeof(double));
  b = malloc(n * sizeof(double));
  partial_sum = malloc(thread_count * sizeof(double));
  pthread_t* thread_handles = malloc(thread_count * sizeof(pthread_t));
  if (a == NULL || b == NULL || partial_sum == NULL || thread_handles == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }

  atomic_init(&atomic_arrived, 0);
  atomic_init(&atomic_generation, 0);
  pthread_mutex_init(&barrier_mutex, NULL);
  pthread_cond_init(&barrier_cond, NULL);
  pthread_barrier_init(&barrier, NULL, (unsigned int)thread_count);

  printf("Normalization: four levels of barrier implementation\n");
  printf("Threads: %ld, Elements: %ld\n\n", thread_count, n);
  printf("%-28s %12s %14s %8s\n", "Version", "Time (ms)", "Sum of b[]",
         "Check");

  run_version("1. No barrier", Thread_no_barrier, thread_handles);
  run_version("2. Atomic spin barrier", Thread_busy_barrier, thread_handles);
  run_version("3. Mutex + cond barrier", Thread_cond_barrier, thread_handles);
  run_version("4. pthread_barrier", Thread_std_barrier, thread_handles);

  printf("\nExpected sum of b[] is exactly 1.000000.\n");
  printf("Version 1 is expected to fail, and to fail differently each run.\n");

  pthread_mutex_destroy(&barrier_mutex);
  pthread_cond_destroy(&barrier_cond);
  pthread_barrier_destroy(&barrier);
  free(a);
  free(b);
  free(partial_sum);
  free(thread_handles);
  return 0;
}

In [ ]:
nm = compile_c(f"{SRC_DIR}/pthread_normalization.c", f"{SRC_DIR}/pthread_normalization")

## 6. 版本一 · 无屏障：阶段错位

传入 `NULL`，阶段之间没有任何同步。下面连续运行 5 次，观察结果。

In [ ]:
print("8 线程，800 万元素，连续运行 5 次（只看无屏障版本）：\n")
for i in range(5):
    out = run_bin(nm, 8, 8_000_000, echo=False)
    for name, t, s, chk in parse_versions(out):
        if name.startswith("1."):
            sv = "inf/nan" if s == float("inf") else f"{s:.6f}"
            print(f"第 {i+1} 次：sum(b) = {sv:>12}   {chk}")

### 结果解读

无屏障版本**每次都失败**，而且**失败的形态各不相同**——有时结果是某个偏离 1 的数值，有时直接是 `inf`。

这正是阶段错位的表现：

- 若某线程在 0 号线程尚未算出 `global_sum` 时就进入阶段 4，它读到的 `global_sum` 可能是**上一轮的残留值**或**部分汇总值**，于是归一化用错了除数，结果偏离 1；
- 更极端地，若它读到的 `global_sum` 恰好是初始的 **0**，则 `b[i] = a[i] / 0` 得到 `inf`，整个求和随之变成 `inf`。

> **注意**：`global_sum` 的累加本身并**没有**数据竞争——只有 0 号线程写它。问题不在于「写冲突」，而在于「**阶段顺序**」：读发生在了写**完成之前**。
>
> 这与实验五的时序错误同源：缺少的不是互斥，而是**顺序保证**。区别在于，实验五是「一个线程等一个线程」，而这里需要「全体线程互相等待」——因此需要屏障。

单核环境下该版本同样会失败，因为线程被分时轮转时，各阶段的交错依然存在。

## 7. 版本二 · 原子自旋屏障

第一种屏障实现，只用 C11 原子操作，不借助任何操作系统阻塞原语。

```c
atomic_long atomic_arrived = 0;     // 已到达的线程数
atomic_long atomic_generation = 0;  // 屏障轮次编号

static void busy_barrier_wait(void) {
  long my_gen = atomic_load(&atomic_generation);       // 记住本轮编号
  long old_count = atomic_fetch_add(&atomic_arrived, 1);  // 原子地 +1

  if (old_count + 1 == thread_count) {                 // 我是最后一个
    atomic_store(&atomic_arrived, 0);                  //   重置计数
    atomic_fetch_add(&atomic_generation, 1);           //   进入下一轮，放行全体
  } else {                                             // 我不是最后一个
    while (atomic_load(&atomic_generation) == my_gen) {
      // 自旋，直到轮次改变。每个等待线程满载占用一个核心。
    }
  }
}
```

`atomic_fetch_add` 是一条硬件原子指令（AArch64 上为 `LDADD`，或 `LDXR`/`STXR` 循环），保证「读—改—写」不可分割——因此这里的计数**不需要额外的互斥量**。

### 7.1 ⚠️ generation 计数器为何不可省略

这是自旋屏障最容易写错的地方。设想去掉 `generation`，只用「到达计数归零」来放行：

```c
// 错误写法：仅凭 arrived 归零判断
if (atomic_fetch_add(&arrived, 1) + 1 == thread_count) {
  atomic_store(&arrived, 0);           // 归零 = 放行信号
} else {
  while (atomic_load(&arrived) != 0) { }  // 等待归零
}
```

这个写法有一个致命缺陷。程序有**三个连续的屏障**，考虑这样的时序：

1. 最后一个线程到达屏障 2，把 `arrived` 归零，放行；
2. 某个跑得快的线程立刻冲过屏障 2、执行完阶段 3，又冲到了**屏障 3**，把 `arrived` 从 0 加成了 1；
3. 此时还有慢线程停在屏障 2 的 `while (arrived != 0)` 上——它看到 `arrived` 已经不是 0 了，于是**误以为屏障 2 还没放行**，永远等下去。

这称为**轮次混淆**（round confusion）。`generation` 为每一轮屏障打上唯一的编号，等待线程判断的是「轮次是否改变」而非「计数是否归零」，从而彻底避免了这一问题。

> 第 11 节的练习会请你亲手去掉 `generation`，观察程序如何挂起。

### 7.2 优缺点

<!--
| | 说明 |
|---|---|
| ✓ 延迟极低 | 全程无内核介入，放行只是一次原子写 |
| ✓ 适合线程数 ≤ 核心数、屏障间隔极短的场景 | 自旋等待的时间很短 |
| ✗ 等待期间 CPU 占用 100% | 每个等待线程满载占用一个核心，空转不做有用功 |
| ✗ 线程数超过核心数时性能急剧恶化 | 自旋线程占着核心，真正该干活的线程却排不上队 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">✓ 延迟极低</td>
      <td style="text-align: left;">全程无内核介入，放行只是一次原子写</td>
    </tr>
    <tr>
      <td style="text-align: left;">✓ 适合线程数 ≤ 核心数、屏障间隔极短的场景</td>
      <td style="text-align: left;">自旋等待的时间很短</td>
    </tr>
    <tr>
      <td style="text-align: left;">✗ 等待期间 CPU 占用 100%</td>
      <td style="text-align: left;">每个等待线程满载占用一个核心，空转不做有用功</td>
    </tr>
    <tr>
      <td style="text-align: left;">✗ 线程数超过核心数时性能急剧恶化</td>
      <td style="text-align: left;">自旋线程占着核心，真正该干活的线程却排不上队</td>
    </tr>
  </tbody>
</table>

第二条缺点尤其重要：在单核机器上，或线程数超过核心数时，自旋屏障会把 CPU 全部耗在空转上，反而可能是**最慢**的版本。第 9 节的实测会印证这一点。

## 8. 版本三 · 条件变量屏障

把自旋等待换成真正的睡眠——用实验六的条件变量。

```c
long cond_counter = 0;
long cond_generation = 0;
pthread_mutex_t barrier_mutex;
pthread_cond_t barrier_cond;

static void cond_barrier_wait(void) {
  pthread_mutex_lock(&barrier_mutex);
  long my_gen = cond_generation;
  ++cond_counter;

  if (cond_counter == thread_count) {       // 我是最后一个
    cond_counter = 0;
    ++cond_generation;
    pthread_cond_broadcast(&barrier_cond);  // 唤醒全部等待者
  } else {                                  // 我不是最后一个
    while (cond_generation == my_gen) {     // while，不是 if
      pthread_cond_wait(&barrier_cond, &barrier_mutex);
    }
  }
  pthread_mutex_unlock(&barrier_mutex);
}
```

结构与自旋版本一一对应：`cond_counter` 对应 `arrived`，`cond_generation` 对应 `generation`。区别只在于**等待方式**——自旋 `while` 换成了 `pthread_cond_wait`，因而等待线程真正进入睡眠、不占用 CPU。

### 8.1 ⚠️ 放行时为何必须用 `broadcast`

屏障放行的语义是「**所有**等待线程都应当继续」。这正好符合实验六 7.4 节给出的判据：

> 唤醒后**所有**线程都能继续 → 用 `broadcast`。

若这里误用 `pthread_cond_signal`，则最后一个线程只会唤醒**一个**等待者，其余线程将永远睡在 `barrier_cond` 上——程序**死锁**。这是条件变量使用中最典型的错误之一。

对比实验六：那里每次只多一个数据项，只有一个消费者能继续，所以用 `signal`；这里放行的是**全体**，所以必须用 `broadcast`。**同一个判据，不同的场景，不同的选择。**

### 8.2 为何仍然需要 `while`

与实验六完全同理，`pthread_cond_wait` 返回不代表条件成立：

- **虚假唤醒**：POSIX 允许 `pthread_cond_wait` 无故返回；
- **`broadcast` 的群体唤醒**：被唤醒的线程要重新竞争互斥量，醒来后必须重新检查 `cond_generation` 是否真的改变了。

因此等待必须写在 `while (cond_generation == my_gen)` 循环里，而非 `if`。

### 8.3 优缺点

<!--
| | 说明 |
|---|---|
| ✓ 等待线程真正睡眠，CPU 占用为零 | 借助操作系统的阻塞原语 |
| ✓ 线程数超过核心数时依然表现良好 | 睡眠的线程让出核心给其他线程 |
| ✗ 存在上下文切换开销 | 每次睡眠与唤醒都要进出内核 |
| ✗ 屏障间隔极短时，开销可能超过计算本身 | 频繁进出内核得不偿失 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">✓ 等待线程真正睡眠，CPU 占用为零</td>
      <td style="text-align: left;">借助操作系统的阻塞原语</td>
    </tr>
    <tr>
      <td style="text-align: left;">✓ 线程数超过核心数时依然表现良好</td>
      <td style="text-align: left;">睡眠的线程让出核心给其他线程</td>
    </tr>
    <tr>
      <td style="text-align: left;">✗ 存在上下文切换开销</td>
      <td style="text-align: left;">每次睡眠与唤醒都要进出内核</td>
    </tr>
    <tr>
      <td style="text-align: left;">✗ 屏障间隔极短时，开销可能超过计算本身</td>
      <td style="text-align: left;">频繁进出内核得不偿失</td>
    </tr>
  </tbody>
</table>

自旋屏障与条件变量屏障恰好互补：**前者省了内核开销但费 CPU，后者省了 CPU 但有内核开销**。该用哪个，取决于屏障的间隔与线程数是否超过核心数。

## 9. 版本四 · POSIX 标准屏障

前两种手写屏障合起来有二十多行，还要小心维护 generation 与重置逻辑。POSIX 提供了标准的屏障 API，把这一切收进一行调用：

```c
#include <pthread.h>

pthread_barrier_t barrier;
pthread_barrier_init(&barrier, NULL, thread_count);   // count = 需到达的线程数

static void std_barrier_wait(void) {
  pthread_barrier_wait(&barrier);                     // 就这一行
}

pthread_barrier_destroy(&barrier);
```

三个函数即为全部接口：

<!--
| 函数 | 说明 | 返回值 |
|---|---|---|
| `pthread_barrier_init` | 初始化屏障，`count` 指定必须到达的线程总数 | 0 成功；`EINVAL` 表示 count 为 0 |
| `pthread_barrier_wait` | 等待全体到达；最后到达者放行 | 恰有一个线程返回 `PTHREAD_BARRIER_SERIAL_THREAD`，其余返回 0 |
| `pthread_barrier_destroy` | 销毁屏障 | 0 成功 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">函数</th>
      <th style="text-align: left;">说明</th>
      <th style="text-align: left;">返回值</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>pthread_barrier_init</code></td>
      <td style="text-align: left;">初始化屏障，<code>count</code> 指定必须到达的线程总数</td>
      <td style="text-align: left;">0 成功；<code>EINVAL</code> 表示 count 为 0</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>pthread_barrier_wait</code></td>
      <td style="text-align: left;">等待全体到达；最后到达者放行</td>
      <td style="text-align: left;">恰有一个线程返回 <code>PTHREAD_BARRIER_SERIAL_THREAD</code>，其余返回 0</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>pthread_barrier_destroy</code></td>
      <td style="text-align: left;">销毁屏障</td>
      <td style="text-align: left;">0 成功</td>
    </tr>
  </tbody>
</table>

它相比手写实现的优势：

- **由 glibc 与内核实现，通常混合了自旋与阻塞两种策略**，能根据竞争强度自动切换——竞争弱时自旋（省内核开销），竞争强时睡眠（省 CPU）；
- **generation 与重置逻辑已内建**，无需自己维护，从根本上杜绝了轮次混淆；
- 代码从二十余行缩减为一行。

> `pthread_barrier_wait` 恰好返回一个 `PTHREAD_BARRIER_SERIAL_THREAD`，这个「被选中的线程」可以用来执行只需做一次的收尾工作（如本实验中 0 号线程的汇总），无需再自己判断 `my_rank == 0`。

> **注意**：屏障属 POSIX 的**可选特性**，个别平台需检查 `_POSIX_BARRIERS` 宏。主流 Linux（含华为鲲鹏所用的 openEuler / Ubuntu）均已支持。

### 四级演进的教学价值

> 无屏障（错误）→ 原子自旋（能用但费 CPU）→ 条件变量（正确且省 CPU）→ 标准 API（工程最优）。
>
> 理解了前三级，才能明白第四级的 `pthread_barrier_wait` 内部究竟为你做了什么、代价在哪里。**知其然，亦知其所以然**——这正是本章选择 Pthreads 而非更高层封装的意义所在。

## 10. 完整运行与性能对比

一次运行即可对比四个版本。它们处理相同的数据、执行相同的计算，唯一的差别是阶段之间使用的屏障。

In [ ]:
import matplotlib.pyplot as plt

NT = max(2, min(8, os.cpu_count() * 2))
N = 8_000_000
N -= N % NT  # 保证能被线程数整除
out = run_bin(nm, NT, N)

### 屏障实现的耗时对比

下面重复多次取平均，比较三个**正确**版本的耗时（无屏障版本结果错误，不纳入性能比较）。

In [ ]:
cn = {
    "1. No barrier": "无屏障（错误）",
    "2. Atomic spin barrier": "原子自旋屏障",
    "3. Mutex + cond barrier": "条件变量屏障",
    "4. pthread_barrier": "标准屏障",
}

REP = 5
acc = {k: [] for k in cn}
for _ in range(REP):
    for name, t, s, chk in parse_versions(run_bin(nm, NT, N, echo=False)):
        acc[name].append(t)
avg = {k: sum(v) / len(v) for k, v in acc.items() if v}

print(f"线程数 {NT}，元素 {N:,}，重复 {REP} 次取平均，{os.cpu_count()} 核\n")
print(f"{'版本':<18}{'平均耗时(ms)':>14}{'校验':>8}")
print("-" * 42)
last = parse_versions(out)
chk_map = {n: c for n, t, s, c in last}
for k in cn:
    print(f"{cn[k]:<18}{avg[k]:>14.3f}{chk_map.get(k, '-'):>8}")

# 只对正确的三个版本作图
good = ["2. Atomic spin barrier", "3. Mutex + cond barrier", "4. pthread_barrier"]
# 图中一律使用英文标签，避免依赖中文字体
en = {
    "2. Atomic spin barrier": "Atomic spin barrier",
    "3. Mutex + cond barrier": "Cond-var barrier",
    "4. pthread_barrier": "pthread_barrier",
}
labels = [en[k] for k in good]
times = [avg[k] for k in good]
colors = ["#E8833A", "#2E7D32", "#295E96"]

fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.bar(labels, times, color=colors)
for b, t in zip(bars, times):
    ax.text(
        b.get_x() + b.get_width() / 2,
        b.get_height(),
        f"{t:.1f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set_ylabel("Average time (ms, lower is better)")
ax.set_title(f"Time of the three correct barrier implementations "
             f"({NT} threads, {os.cpu_count()} cores)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### 结果解读

**① 自旋屏障在此环境下往往最慢。** 只要线程数超过核心数，自旋等待的线程就会占着核心空转，挤占本该干活的线程。在单核环境中，这一效应最为极端——所有等待线程都在无谓地消耗那唯一的核心。

**② 条件变量屏障与标准屏障表现接近。** 二者都让等待线程睡眠，把核心让给需要的线程。标准屏障通常略优，因为它的自旋/阻塞混合策略能在竞争弱时省去内核开销。

**③ 谁快谁慢取决于环境。** 若线程数远小于核心数、且屏障间隔极短，自旋屏障反而可能最快——因为它省去了进出内核的开销。**没有绝对最优的屏障，只有与场景匹配的屏障。**

> 这与实验六的结论一脉相承：同步原语的性能高度依赖负载特征。
> 工程上的稳妥选择是 `pthread_barrier_wait`——它把自旋与阻塞的权衡交给了运行库，在多数场景下都不会太差。

### 关于本机结果

若在单核或核心数少于线程数的环境运行，自旋屏障的劣势会被放大，这是符合预期的。请在鲲鹏多核平台上，用**线程数不超过核心数**的配置重跑本节，以观察三者更接近真实的相对表现。

## 11. 结果分析

本实验引入了「全体互相等待」这一新的同步形态，并建立了三项认识：

**① 互斥不能替代同步。** 保护了共享变量的累加，不等于保证了阶段的先后顺序。前者用互斥量，后者用屏障，二者不可互换。这是版本一失败的根本原因。

**② 屏障可以手写，其内部并不神秘。** 一个正确的屏障只需两个要素：一个**到达计数器**，和一个**轮次编号**（generation）。前者判断「是否全部到齐」，后者避免「轮次混淆」。理解了这两点，就理解了 `pthread_barrier_wait` 的核心。

**③ 等待方式的选择是一种权衡。** 自旋省内核开销但费 CPU，阻塞省 CPU 但有内核开销。标准屏障通过混合策略在二者间自动权衡，因而是工程首选。

### 🎓 结论

四级屏障的演进，浓缩了本章的一条核心方法：**先理解问题的本质，再选择合适的工具**。

<!--
| 版本 | 正确性 | 等待方式 | CPU（等待时） | 适用场景 |
|---|---|---|---|---|
| 无屏障 | **错误** | — | — | 无（反面教材） |
| 原子自旋 | 正确 | 自旋 | 100% | 线程数 ≤ 核心数、屏障极密 |
| 条件变量 | 正确 | 睡眠 | 0% | 线程数 > 核心数 |
| 标准屏障 | 正确 | 混合 | 视竞争而定 | **通用首选** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">正确性</th>
      <th style="text-align: left;">等待方式</th>
      <th style="text-align: left;">CPU（等待时）</th>
      <th style="text-align: left;">适用场景</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">无屏障</td>
      <td style="text-align: left;"><strong>错误</strong></td>
      <td style="text-align: left;">—</td>
      <td style="text-align: left;">—</td>
      <td style="text-align: left;">无（反面教材）</td>
    </tr>
    <tr>
      <td style="text-align: left;">原子自旋</td>
      <td style="text-align: left;">正确</td>
      <td style="text-align: left;">自旋</td>
      <td style="text-align: left;">100%</td>
      <td style="text-align: left;">线程数 ≤ 核心数、屏障极密</td>
    </tr>
    <tr>
      <td style="text-align: left;">条件变量</td>
      <td style="text-align: left;">正确</td>
      <td style="text-align: left;">睡眠</td>
      <td style="text-align: left;">0%</td>
      <td style="text-align: left;">线程数 &gt; 核心数</td>
    </tr>
    <tr>
      <td style="text-align: left;">标准屏障</td>
      <td style="text-align: left;">正确</td>
      <td style="text-align: left;">混合</td>
      <td style="text-align: left;">视竞争而定</td>
      <td style="text-align: left;"><strong>通用首选</strong></td>
    </tr>
  </tbody>
</table>

> 屏障还引出一个更深的问题：**它的等待时间由最慢的线程决定**（第 2 节）。
> 这意味着，屏障密集型算法的性能上限，取决于**负载是否均衡**——再高效的屏障实现，也无法弥补线程之间的负载失衡。这一点将在后续的性能优化中反复出现。

## 12. 🔧 动手练习

请修改代码、重新编译并运行，观察行为的变化：

1. 从 `busy_barrier_wait` 中去掉 `generation` 逻辑，改为仅凭 `atomic_arrived` 归零来放行等待线程。用 8 线程运行，观察程序是否挂起，并结合 7.1 节的时序分析解释「轮次混淆」是如何发生的。
2. 把 `cond_barrier_wait` 中的 `pthread_cond_broadcast` 改为 `pthread_cond_signal`，用 4 线程运行，观察程序是否挂起，并解释为什么屏障放行必须唤醒全部线程。
3. 把 `cond_barrier_wait` 中的 `while (cond_generation == my_gen)` 改为 `if`，多次运行观察是否偶发出错，并说明 `while` 在此处防范的是哪一类情形。
4. 故意让各线程负载不均：将数据划分改为「0 号线程分到一半元素，其余线程平分另一半」，测量各屏障版本的耗时变化，并结合第 2 节「屏障等待由最慢线程决定」解释所得结果。
5. 用第 10 节的方法测量三种屏障在 1、2、4、8、16 线程下的耗时曲线，找出自旋屏障在本机由「较快」转为「较慢」的线程数临界点，并说明它与核心数的关系。

## 13. 🤔 思考题

- 版本一的 `global_sum` 只由 0 号线程写入，不存在数据竞争。既然如此，它为什么仍然出错？这与实验五消息传递中的时序错误有何异同？
- 自旋屏障用 `atomic_fetch_add` 对到达计数做原子递增。若改用「普通变量 + 互斥量」来计数，还能正确工作吗？与条件变量屏障相比，这样做少了什么、多了什么？
- 条件变量屏障中，最后一个线程先 `broadcast` 再 `unlock`。若把这两步调换（先 `unlock` 再 `broadcast`），程序是否仍然正确？可能带来什么问题？
- `pthread_barrier_wait` 恰好让一个线程返回 `PTHREAD_BARRIER_SERIAL_THREAD`。本实验的阶段 3 只需 0 号线程执行汇总。若改用这个返回值来挑选执行汇总的线程，代码会有什么变化？这样做有什么好处？
- 本实验有三个屏障、四个阶段。能否通过重新组织算法来减少屏障的数量？例如，阶段 3 的「汇总」是否一定要由单个线程串行完成，还是可以用并行归约配合更少的屏障来实现？
- 屏障的等待时间由最慢的线程决定。假设一个算法有 10 个阶段，每个阶段各线程的负载都略有波动（有时这个线程慢，有时那个线程慢）。随着阶段数增加，屏障累积的空等时间会如何变化？这对「细粒度分阶段」的算法设计有何启示？

## 14. 小结与后续

本实验通过同一个屏障的四种实现，完整呈现了「全体互相等待」这一同步形态：

<!--
| 版本 | 屏障实现 | 新增知识点 |
|---|---|---|
| **一 · 无屏障** | — | 互斥 ≠ 同步、阶段错位错误 |
| **二 · 原子自旋** | 原子计数 + 自旋 | generation 计数器、轮次混淆 |
| **三 · 条件变量** | 互斥量 + 条件变量 | 屏障放行必须用 `broadcast` |
| **四 · 标准屏障** | `pthread_barrier_t` | 标准 API 的取舍、自旋/阻塞混合策略 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">屏障实现</th>
      <th style="text-align: left;">新增知识点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>一 · 无屏障</strong></td>
      <td style="text-align: left;">—</td>
      <td style="text-align: left;">互斥 ≠ 同步、阶段错位错误</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>二 · 原子自旋</strong></td>
      <td style="text-align: left;">原子计数 + 自旋</td>
      <td style="text-align: left;">generation 计数器、轮次混淆</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>三 · 条件变量</strong></td>
      <td style="text-align: left;">互斥量 + 条件变量</td>
      <td style="text-align: left;">屏障放行必须用 <code>broadcast</code></td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>四 · 标准屏障</strong></td>
      <td style="text-align: left;"><code>pthread_barrier_t</code></td>
      <td style="text-align: left;">标准 API 的取舍、自旋/阻塞混合策略</td>
    </tr>
  </tbody>
</table>

本实验也把实验六的条件变量派上了新用场——用它亲手实现了屏障，印证了「条件变量是更通用的构造」这一说法。至此，本章模块五的核心同步原语已全部就位。

➡️ **后续内容：实验八 并发链表：读多写少场景的锁策略**。本章前几个实验处理的都是**规则的数组**，数据划分清晰、无锁竞争或竞争可控。实验八转向**动态数据结构**——一个有序链表，支持查找、插入、删除。这里的挑战不同以往：操作会改变结构本身，且查找操作往往占绝大多数。届时将看到一把大锁如何扼杀并发，以及**交接锁**（hand-over-hand locking）与**读写锁**如何在读多写少的场景下逼近理想的并发度。其中交接锁按节点物理顺序逐一加锁，正是实验四资源分级规范的又一次应用。